# 33｜手写 TextCNN 与 Hierarchical Attention Network 文档分类

同一个文档分类任务可以有两种完全不同的归纳偏置：TextCNN 用多尺度卷积抓局部 n-gram；HAN 先在句内编码词，再在篇章内编码句子，并输出两级可审计注意力。本笔记不调用 `nn.RNN/LSTM/GRU`，而是手写 GRU cell、双向扫描、masked attention、`forward`、训练与可信制品。

> 合成数据只证明实现可以学习一个已知规则，不代表真实文档分类效果或注意力具有因果解释性。

## 1. 两套输入合同

- TextCNN：`tokens/mask: [B,T]`，有效 token 为左对齐前缀；长度必须不小于最大卷积核。
- HAN：`tokens/word_mask: [B,S,W]`；有效句子左对齐，每个有效句内部的词也左对齐。padding 句允许全空，但每篇文档至少一个有效句。
- TextCNN 输出 `[B,C]`；HAN 额外返回 `word_alpha: [B,S,W]` 和 `sent_alpha: [B,S]`。
- 有效 attention 权重和为 1，padding 句/词权重为 0；全 padding 文档必须 fail closed。
- 固定 CPU 和随机种子，不联网，不调用任何预制循环层。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

import copy
import hashlib
import io
import json
import random

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260812
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
PAD = 0
VOCAB_SIZE = 40
NUM_CLASSES = 3
CLASSIFIER_TOKENS = ["<pad>"] + [f"token_{index}" for index in range(1, VOCAB_SIZE)]

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
assert VOCAB_SIZE > 10 and NUM_CLASSES == 3
assert len(CLASSIFIER_TOKENS) == VOCAB_SIZE and len(set(CLASSIFIER_TOKENS)) == VOCAB_SIZE
print({"torch": torch.__version__, "seed": SEED})

## 2. 统一验证变长与空 padding

长度错误若拖到卷积、softmax 或 gather 才暴露，报错通常很难定位。我们先验证 prefix mask：`True,True,False` 合法，`True,False,True` 非法。HAN 的 padding 句可以全 False，但有效句之后不能再次出现有效句。全空文档没有可归一化的句级分布，因此立即拒绝。

In [ ]:
def validate_prefix_rows(mask, allow_empty=False, name="mask"):
    if mask.dtype != torch.bool or mask.ndim != 2:
        raise ValueError(f"{name} 必须是二维 bool")
    if not allow_empty and bool((~mask.any(1)).any()):
        raise ValueError(f"{name} 每行至少一个有效位置")
    if mask.shape[1] > 1 and bool((mask[:, 1:] & ~mask[:, :-1]).any()):
        raise ValueError(f"{name} 必须是左对齐前缀")


def masked_softmax(scores, mask):
    if scores.shape != mask.shape or mask.dtype != torch.bool:
        raise ValueError("masked_softmax 形状或类型错误")
    masked_scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    row_has_value = mask.any(-1, keepdim=True)
    safe_max = torch.where(row_has_value, masked_scores.max(-1, keepdim=True).values, torch.zeros_like(scores[:, :1]))
    numer = torch.exp(masked_scores - safe_max) * mask
    weights = numer / numer.sum(-1, keepdim=True).clamp_min(torch.finfo(scores.dtype).tiny)
    return weights

scores0 = torch.tensor([[1.0, 2.0, 99.0], [3.0, 4.0, 5.0]])
mask_test = torch.tensor([[True, True, False], [False, False, False]])
w0 = masked_softmax(scores0, mask_test)
assert torch.allclose(w0[0].sum(), torch.tensor(1.0))
assert torch.count_nonzero(w0[0, 2:]) == 0 and torch.count_nonzero(w0[1]) == 0

## 3. TextCNN：多尺度 n-gram 与 max-over-time

对窗口宽度 $k$：

$$c_i^{(k)}=\operatorname{ReLU}(W_k\,x_{i:i+k-1}+b_k),\qquad
\hat c^{(k)}=\max_{i\in\text{valid windows}}c_i^{(k)}.$$

多个 `k` 的 pooled feature 拼接后分类。关键不是 `Conv1d` 本身，而是**只允许全部由有效 token 构成的窗口参与最大值**；否则 padding embedding 或卷积 bias 会成为长度特征。

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, channels, kernel_sizes, num_classes):
        super().__init__()
        if not kernel_sizes or min(kernel_sizes) <= 0 or len(set(kernel_sizes)) != len(kernel_sizes):
            raise ValueError("kernel_sizes 必须是非空正整数集合")
        self.config = dict(vocab_size=vocab_size, embed_dim=embed_dim, channels=channels,
                           kernel_sizes=list(kernel_sizes), num_classes=num_classes)
        self.max_kernel = max(kernel_sizes)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, channels, k) for k in kernel_sizes])
        self.classifier = nn.Linear(channels * len(kernel_sizes), num_classes)

    def forward(self, tokens, mask):
        if tokens.ndim != 2 or tokens.dtype != torch.long or mask.shape != tokens.shape:
            raise ValueError("TextCNN tokens/mask 合同错误")
        validate_prefix_rows(mask, allow_empty=False, name="TextCNN mask")
        lengths = mask.sum(1)
        if bool((lengths < self.max_kernel).any()):
            raise ValueError("有效长度不能小于最大卷积核")
        if tokens.numel() and (int(tokens.min()) < 0 or int(tokens.max()) >= self.config["vocab_size"]):
            raise ValueError("token id 越界")
        embedded = self.embedding(tokens).transpose(1, 2)
        pooled = []
        for conv, kernel in zip(self.convs, self.config["kernel_sizes"]):
            features = F.relu(conv(embedded))
            valid_windows = mask.unfold(1, kernel, 1).all(-1)
            features = features.masked_fill(~valid_windows.unsqueeze(1), torch.finfo(features.dtype).min)
            pooled.append(features.max(-1).values)
        document_features = torch.cat(pooled, dim=-1)
        return self.classifier(document_features), document_features

textcnn_probe = TextCNN(VOCAB_SIZE, 12, 8, [2, 3, 4], NUM_CLASSES)
tc_tokens = torch.tensor([[4, 5, 6, 7, PAD, PAD], [8, 9, 10, 11, 12, PAD]])
tc_mask = tc_tokens.ne(PAD)
tc_logits, tc_features = textcnn_probe(tc_tokens, tc_mask)
assert tc_logits.shape == (2, NUM_CLASSES) and tc_features.shape == (2, 24)

try:
    textcnn_probe(torch.tensor([[4, 5, PAD, PAD]]), torch.tensor([[True, True, False, False]]))
    raise AssertionError("过短序列应被拒绝")
except ValueError as exc:
    assert "最大卷积核" in str(exc)

## 4. 手写 GRU cell 与双向变长扫描

使用更新门、重置门和候选状态：

$$z_t=\sigma(W_zx_t+U_zh_{t-1}),\quad r_t=\sigma(W_rx_t+U_rh_{t-1}),$$
$$\tilde h_t=\tanh(W_nx_t+r_t\odot U_nh_{t-1}),\quad
h_t=z_t\odot h_{t-1}+(1-z_t)\odot\tilde h_t.$$

双向扫描各用一套 cell。padding 步不更新状态且输出清零；反向扫描仍按原 mask 判断每个位置，而不是把 padding 当作序列开头。

In [ ]:
class ScratchGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        if input_size <= 0 or hidden_size <= 0:
            raise ValueError("GRU 维度必须为正")
        self.input_size, self.hidden_size = input_size, hidden_size
        self.x_proj = nn.Linear(input_size, 3 * hidden_size, bias=True)
        self.h_proj = nn.Linear(hidden_size, 3 * hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        if x_t.ndim != 2 or h_prev.ndim != 2 or x_t.shape[0] != h_prev.shape[0]:
            raise ValueError("GRU 单步输入形状错误")
        if x_t.shape[1] != self.input_size or h_prev.shape[1] != self.hidden_size:
            raise ValueError("GRU 特征维错误")
        x_z, x_r, x_n = self.x_proj(x_t).chunk(3, -1)
        h_z, h_r, h_n = self.h_proj(h_prev).chunk(3, -1)
        z = torch.sigmoid(x_z + h_z)
        r = torch.sigmoid(x_r + h_r)
        candidate = torch.tanh(x_n + r * h_n)
        return z * h_prev + (1.0 - z) * candidate


class ScratchBiGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.forward_cell = ScratchGRUCell(input_size, hidden_size)
        self.backward_cell = ScratchGRUCell(input_size, hidden_size)

    def _scan(self, x, mask, cell, reverse=False):
        B, T, _ = x.shape
        h = x.new_zeros(B, self.hidden_size)
        outputs = [None] * T
        indices = range(T - 1, -1, -1) if reverse else range(T)
        for t in indices:
            proposed = cell(x[:, t], h)
            active = mask[:, t:t + 1]
            h = torch.where(active, proposed, h)
            outputs[t] = torch.where(active, h, torch.zeros_like(h))
        return torch.stack(outputs, dim=1)

    def forward(self, x, mask):
        if x.ndim != 3 or mask.shape != x.shape[:2]:
            raise ValueError("BiGRU 输入形状错误")
        validate_prefix_rows(mask, allow_empty=True, name="BiGRU mask")
        forward = self._scan(x, mask, self.forward_cell, reverse=False)
        backward = self._scan(x, mask, self.backward_cell, reverse=True)
        return torch.cat([forward, backward], dim=-1)

bigru_probe = ScratchBiGRU(5, 4)
x_probe = torch.randn(2, 4, 5)
m_probe = torch.tensor([[True, True, False, False], [False, False, False, False]])
h_probe = bigru_probe(x_probe, m_probe)
assert h_probe.shape == (2, 4, 8)
assert torch.count_nonzero(h_probe[0, 2:]) == 0 and torch.count_nonzero(h_probe[1]) == 0

## 5. HAN：词级与句级两次 attention

加性注意力先计算 $u_i=\tanh(Wh_i+b)$、$e_i=v^\top u_i$，再仅对有效位置 softmax：

$$\alpha_i=\frac{\exp(e_i)}{\sum_{j\in valid}\exp(e_j)},\qquad s=\sum_i\alpha_i h_i.$$

第一层把词表示聚合成句向量，第二层把句向量聚合成文档向量。padding 句的词权重总和为 0；有效句为 1；每篇有效文档的句权重为 1。注意力权重是模型内部路由，不自动等于因果解释。

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, input_dim, attention_dim):
        super().__init__()
        self.proj = nn.Linear(input_dim, attention_dim)
        self.context = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, states, mask):
        if states.ndim != 3 or mask.shape != states.shape[:2]:
            raise ValueError("attention 输入形状错误")
        scores = self.context(torch.tanh(self.proj(states))).squeeze(-1)
        weights = masked_softmax(scores, mask)
        pooled = torch.sum(states * weights.unsqueeze(-1), dim=1)
        return pooled, weights


class HierarchicalAttentionNetwork(nn.Module):
    def __init__(self, vocab_size, embed_dim, word_hidden, sentence_hidden, attention_dim, num_classes):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, embed_dim=embed_dim, word_hidden=word_hidden,
                           sentence_hidden=sentence_hidden, attention_dim=attention_dim,
                           num_classes=num_classes)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.word_encoder = ScratchBiGRU(embed_dim, word_hidden)
        self.word_attention = AdditiveAttention(2 * word_hidden, attention_dim)
        self.sentence_encoder = ScratchBiGRU(2 * word_hidden, sentence_hidden)
        self.sentence_attention = AdditiveAttention(2 * sentence_hidden, attention_dim)
        self.classifier = nn.Linear(2 * sentence_hidden, num_classes)

    def forward(self, tokens, word_mask):
        if tokens.ndim != 3 or tokens.dtype != torch.long or word_mask.shape != tokens.shape:
            raise ValueError("HAN tokens/word_mask 合同错误")
        if tokens.numel() and (int(tokens.min()) < 0 or int(tokens.max()) >= self.config["vocab_size"]):
            raise ValueError("token id 越界")
        B, S, W = tokens.shape
        flat_mask = word_mask.reshape(B * S, W)
        validate_prefix_rows(flat_mask, allow_empty=True, name="word mask")
        sentence_mask = word_mask.any(-1)
        validate_prefix_rows(sentence_mask, allow_empty=False, name="sentence mask")

        embedded = self.embedding(tokens).reshape(B * S, W, -1)
        word_states = self.word_encoder(embedded, flat_mask)
        sentence_vectors, word_alpha = self.word_attention(word_states, flat_mask)
        sentence_vectors = sentence_vectors.reshape(B, S, -1)
        sentence_states = self.sentence_encoder(sentence_vectors, sentence_mask)
        document_vector, sentence_alpha = self.sentence_attention(sentence_states, sentence_mask)
        logits = self.classifier(document_vector)
        return {"logits": logits, "document": document_vector,
                "word_alpha": word_alpha.reshape(B, S, W), "sentence_alpha": sentence_alpha,
                "sentence_mask": sentence_mask}

In [ ]:
han_probe = HierarchicalAttentionNetwork(VOCAB_SIZE, 10, 6, 7, 8, NUM_CLASSES)
docs_probe = torch.tensor([
    [[4, 5, 6, PAD], [7, 8, PAD, PAD], [PAD, PAD, PAD, PAD]],
    [[9, 10, 11, 12], [PAD, PAD, PAD, PAD], [PAD, PAD, PAD, PAD]],
])
word_mask_probe = docs_probe.ne(PAD)
han_out = han_probe(docs_probe, word_mask_probe)
word_sums = han_out["word_alpha"].sum(-1)
sent_sums = han_out["sentence_alpha"].sum(-1)

assert han_out["logits"].shape == (2, NUM_CLASSES)
assert torch.allclose(word_sums[han_out["sentence_mask"]], torch.ones_like(word_sums[han_out["sentence_mask"]]))
assert torch.count_nonzero(word_sums[~han_out["sentence_mask"]]) == 0
assert torch.allclose(sent_sums, torch.ones_like(sent_sums))
assert torch.count_nonzero(han_out["sentence_alpha"][~han_out["sentence_mask"]]) == 0

all_pad = torch.zeros(1, 2, 3, dtype=torch.long)
try:
    han_probe(all_pad, all_pad.ne(PAD))
    raise AssertionError("全 padding 文档应被拒绝")
except ValueError as exc:
    assert "sentence mask" in str(exc)

## 6. 同一批合成文档，两种模型公平对照

三类文档分别含有类标记 token `4/5/6`，其余为干扰词；句数与句长变化。我们从层级表示生成 TextCNN 的扁平前缀，并保持同一标签。数据切分在这里只用于受控过拟合；真实评估应按作者、主题、时间或文档来源切分，防止近重复文本泄漏。

In [ ]:
def make_documents(n_per_class=5, max_sentences=3, max_words=5):
    docs, labels = [], []
    rng = random.Random(SEED + 20)
    for label in range(NUM_CLASSES):
        marker = 4 + label
        for sample_index in range(n_per_class):
            n_sentences = 1 + (sample_index % max_sentences)
            doc = []
            for s in range(n_sentences):
                length = 4 + ((sample_index + s) % 2)
                words = [7 + rng.randrange(VOCAB_SIZE - 7) for _ in range(length)]
                if s == sample_index % n_sentences:
                    words[(sample_index + s) % length] = marker
                doc.append(words + [PAD] * (max_words - length))
            doc += [[PAD] * max_words for _ in range(max_sentences - n_sentences)]
            docs.append(doc); labels.append(label)
    return torch.tensor(docs, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


def flatten_documents(docs):
    rows = []
    for doc in docs.tolist():
        valid = [token for sentence in doc for token in sentence if token != PAD]
        rows.append(valid)
    max_len = max(len(row) for row in rows)
    return torch.tensor([row + [PAD] * (max_len - len(row)) for row in rows], dtype=torch.long)

documents, document_labels = make_documents()
document_word_mask = documents.ne(PAD)
flat_documents = flatten_documents(documents)
flat_mask = flat_documents.ne(PAD)
assert documents.shape == (15, 3, 5) and document_labels.shape == (15,)
assert int(flat_mask.sum()) == int(document_word_mask.sum())
assert int(flat_mask.sum(1).min()) >= 4

## 7. 梯度合同与受控过拟合

训练前后在完全相同的 15 篇文档上比较交叉熵。我们分别检查 embedding、卷积/GRU、attention 和分类器能收到有限非零梯度。小集合训练准确率高只是一条“电路连通”证据；模型选择仍必须使用未参与优化的验证集。

In [ ]:
def train_to_overfit(model, batch_fn, labels, steps=120, lr=0.025):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    with torch.no_grad():
        initial_logits = batch_fn(model)
        initial_loss = float(F.cross_entropy(initial_logits, labels))
    first_gradients = None
    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)
        logits = batch_fn(model)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        if step == 0:
            first_gradients = {
                name: float(parameter.grad.abs().sum())
                for name, parameter in model.named_parameters()
                if parameter.requires_grad and parameter.grad is not None
            }
            assert first_gradients and sum(first_gradients.values()) > 0
            assert all(torch.isfinite(parameter.grad).all() for parameter in model.parameters() if parameter.grad is not None)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
    model.eval()
    with torch.no_grad():
        final_logits = batch_fn(model)
        final_loss = float(F.cross_entropy(final_logits, labels))
        accuracy = float((final_logits.argmax(-1) == labels).float().mean())
    return initial_loss, final_loss, accuracy, first_gradients


torch.manual_seed(SEED + 1)
textcnn = TextCNN(VOCAB_SIZE, embed_dim=12, channels=10, kernel_sizes=[2, 3, 4], num_classes=NUM_CLASSES)
tc_metrics = train_to_overfit(
    textcnn, lambda model: model(flat_documents, flat_mask)[0], document_labels, steps=70
)

torch.manual_seed(SEED + 2)
han = HierarchicalAttentionNetwork(VOCAB_SIZE, embed_dim=10, word_hidden=7,
                                   sentence_hidden=7, attention_dim=8, num_classes=NUM_CLASSES)
han_metrics = train_to_overfit(
    han, lambda model: model(documents, document_word_mask)["logits"], document_labels, steps=90
)

assert tc_metrics[1] < tc_metrics[0] * 0.15 and tc_metrics[2] == 1.0
assert han_metrics[1] < han_metrics[0] * 0.15 and han_metrics[2] == 1.0
for prefix in ["embedding", "convs", "classifier"]:
    assert sum(value for name, value in tc_metrics[3].items() if name.startswith(prefix)) > 0, prefix
for prefix in ["embedding", "word_encoder", "word_attention", "sentence_encoder", "sentence_attention", "classifier"]:
    assert sum(value for name, value in han_metrics[3].items() if name.startswith(prefix)) > 0, prefix
print({"TextCNN": [round(x, 4) for x in tc_metrics[:3]],
       "HAN": [round(x, 4) for x in han_metrics[:3]]})

## 8. 评估与解释边界

- 分类：accuracy、macro-F1、逐类 recall、混淆矩阵，并按文档长度、语言、来源切片。
- 稳健性：追加 padding、删除无关句、同义改写、标点/Unicode 归一化变化；这些不能与标签规则共享生成模板。
- 效率：TextCNN 各卷积约 $O(BT\,KEDC)$；HAN 的循环扫描约 $O(BSW(H^2+EH)+BSH_s^2)$，难像卷积一样完全并行。
- 注意力可用于调试“模型读了哪里”，但高权重不证明该词对预测具有因果贡献；需要遮挡、反事实或梯度方法交叉验证。

In [ ]:
textcnn.eval(); han.eval()
with torch.no_grad():
    trained_han_out = han(documents, document_word_mask)
    trained_word_sum = trained_han_out["word_alpha"].sum(-1)
    trained_sent_sum = trained_han_out["sentence_alpha"].sum(-1)
    valid_sentences = trained_han_out["sentence_mask"]
assert torch.allclose(trained_word_sum[valid_sentences], torch.ones_like(trained_word_sum[valid_sentences]), atol=1e-6)
assert torch.count_nonzero(trained_word_sum[~valid_sentences]) == 0
assert torch.allclose(trained_sent_sum, torch.ones_like(trained_sent_sum), atol=1e-6)

# 在固定 mask 下更改 padding token id，不得影响 HAN 的 logits。
pad_ids_changed = documents.clone()
pad_ids_changed[~document_word_mask] = 17
with torch.no_grad():
    original_logits = han(documents, document_word_mask)["logits"]
    changed_logits = han(pad_ids_changed, document_word_mask)["logits"]
assert torch.equal(original_logits, changed_logits)

bad_mask = torch.tensor([[True, False, True, False]])
try:
    validate_prefix_rows(bad_mask, name="bad mask")
    raise AssertionError("带洞 mask 应被拒绝")
except ValueError as exc:
    assert "左对齐" in str(exc)


# GRU 门数值 oracle：所有投影为 0 时 z=r=0.5、candidate=0，故 h_new=0.5*h_prev。
gate_oracle = ScratchGRUCell(3, 2).eval()
with torch.no_grad():
    gate_oracle.x_proj.weight.zero_()
    gate_oracle.x_proj.bias.zero_()
    gate_oracle.h_proj.weight.zero_()
gate_h = torch.tensor([[2.0, -4.0], [1.0, 3.0]])
gate_out = gate_oracle(torch.randn(2, 3), gate_h)
assert torch.equal(gate_out, 0.5 * gate_h)

# 反向扫描不能读取 padding embedding；TextCNN 也只能池化全有效窗口。
reverse_oracle = ScratchBiGRU(5, 4).eval()
reverse_x = torch.randn(1, 5, 5)
reverse_mask = torch.tensor([[True, True, True, False, False]])
reverse_changed = reverse_x.clone()
reverse_changed[:, 3:] = 999.0
with torch.no_grad():
    reverse_a = reverse_oracle(reverse_x, reverse_mask)
    reverse_b = reverse_oracle(reverse_changed, reverse_mask)
assert torch.equal(reverse_a, reverse_b)

flat_pad_changed = flat_documents.clone()
flat_pad_changed[~flat_mask] = VOCAB_SIZE - 1
with torch.no_grad():
    textcnn_a = textcnn(flat_documents, flat_mask)[0]
    textcnn_b = textcnn(flat_pad_changed, flat_mask)[0]
assert torch.equal(textcnn_a, textcnn_b)

# 单独的 additive attention 对 state 置换保持 pooled 不变，weights 随位置同步置换。
permutation_attention = AdditiveAttention(6, 4).eval()
permutation_states = torch.randn(2, 3, 6)
permutation_mask = torch.ones(2, 3, dtype=torch.bool)
permutation = torch.tensor([2, 0, 1])
with torch.no_grad():
    pooled_a, alpha_a = permutation_attention(permutation_states, permutation_mask)
    pooled_b, alpha_b = permutation_attention(
        permutation_states[:, permutation], permutation_mask[:, permutation]
    )
assert torch.allclose(pooled_a, pooled_b, atol=1e-6)
assert torch.allclose(alpha_b, alpha_a[:, permutation], atol=1e-6)

# 完整 HAN 含句级 BiGRU，应对句序敏感；空句夹在有效句之间必须拒绝。
two_sentence_doc = documents[1:2]
two_sentence_mask = document_word_mask[1:2]
with torch.no_grad():
    order_a = han(two_sentence_doc, two_sentence_mask)["logits"]
    order_b = han(two_sentence_doc[:, [1, 0, 2]], two_sentence_mask[:, [1, 0, 2]])["logits"]
sentence_order_delta = float((order_a - order_b).abs().max())
assert sentence_order_delta > 1e-6
middle_empty = documents[2:3].clone()
middle_empty[:, 1] = PAD
try:
    han(middle_empty, middle_empty.ne(PAD))
    raise AssertionError("有效句之间的空句必须被拒绝")
except ValueError as exc:
    assert "sentence mask" in str(exc)
print({"gru_gate_error": float((gate_out - 0.5 * gate_h).abs().max()),
       "reverse_padding_error": float((reverse_a - reverse_b).abs().max()),
       "textcnn_padding_error": float((textcnn_a - textcnn_b).abs().max()),
       "han_sentence_order_delta": sentence_order_delta})


## 9. 完整双模型制品与发布者 registry

TextCNN 与 HAN 必须分别绑定架构白名单、构造参数、**完整 token 顺序**、唯一且非空的 label map、各自输入表示与 padding 规则、实际训练文档/扁平输入/split、训练 recipe，以及权重的原始字节和 tensor 语义摘要。

package 内自带 SHA 不是信任根。下面由发布者侧只读 registry 保存 `(artifact_id, version) -> immutable manifest digest`；loader 先查 registry，再校验 state bytes，以及包含 tensor key/dtype/shape/bytes 的摘要。攻击者即使整体替换模型、词表或标签并重算内部 hash，也不能更新 registry。

In [ ]:
from types import MappingProxyType

def json_hash(value):
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()


def classifier_tensor_state_hash(state_dict):
    digest = hashlib.sha256()
    for key in sorted(state_dict):
        tensor = state_dict[key]
        if not isinstance(tensor, torch.Tensor):
            raise TypeError("state_dict 只能包含 tensor")
        value = tensor.detach().cpu().contiguous()
        descriptor = {"key": key, "dtype": str(value.dtype), "shape": list(value.shape)}
        digest.update(json.dumps(descriptor, sort_keys=True, separators=(",", ":")).encode())
        digest.update(value.numpy().tobytes(order="C"))
    return digest.hexdigest()


def expected_preprocess(architecture, config):
    common = {"tokenizer": "synthetic-id-list-v1", "normalization": "identity-demo-v1",
              "padding": "right", "pad_id": PAD}
    if architecture == "textcnn":
        return {**common, "representation": "flatten-row-major-valid-tokens",
                "kernel_sizes": list(config["kernel_sizes"]),
                "min_valid_tokens": max(config["kernel_sizes"])}
    if architecture == "han":
        return {**common, "representation": "hierarchical-document",
                "max_sentences": 3, "max_words": 5,
                "empty_sentence_policy": "trailing-only", "word_mask_true": "valid"}
    raise ValueError("未知架构")


def validate_classifier_manifest(manifest):
    required = {"schema", "artifact_id", "version", "subject", "architecture", "config",
                "vocab", "label_map", "preprocess", "training_snapshot",
                "state_bytes_sha256", "state_tensor_sha256"}
    if not isinstance(manifest, dict) or set(manifest) != required:
        raise ValueError("classifier manifest 字段不完整")
    if manifest["schema"] != "document-classifier-v2" or not manifest["artifact_id"] or not manifest["version"]:
        raise ValueError("artifact 身份字段错误")
    architecture, config = manifest["architecture"], manifest["config"]
    expected_config_keys = {
        "textcnn": {"vocab_size", "embed_dim", "channels", "kernel_sizes", "num_classes"},
        "han": {"vocab_size", "embed_dim", "word_hidden", "sentence_hidden",
                "attention_dim", "num_classes"},
    }
    if architecture not in expected_config_keys or set(config) != expected_config_keys[architecture]:
        raise ValueError("架构或 config 字段错误")
    vocab = manifest["vocab"]
    if set(vocab) != {"kind", "tokens", "pad_id"} or vocab["kind"] != "synthetic-id-list-v1":
        raise ValueError("vocab schema 错误")
    tokens = vocab["tokens"]
    if not isinstance(tokens, list) or len(tokens) != len(set(tokens)):
        raise ValueError("完整 token 顺序必须唯一")
    if len(tokens) != config["vocab_size"] or vocab["pad_id"] != PAD or tokens[PAD] != "<pad>":
        raise ValueError("vocab_size/pad/token 顺序不一致")
    labels = manifest["label_map"]
    if (not isinstance(labels, list) or len(labels) != config["num_classes"] or
            len(labels) != len(set(labels)) or any(not isinstance(label, str) or not label for label in labels)):
        raise ValueError("label 数量必须匹配 num_classes，且唯一非空")
    if manifest["preprocess"] != expected_preprocess(architecture, config):
        raise ValueError("预处理快照与架构不一致")
    snapshot = manifest["training_snapshot"]
    if snapshot.get("vocab_sha256") != json_hash(vocab):
        raise ValueError("训练快照 vocab 指纹错误")
    if snapshot.get("label_map_sha256") != json_hash(labels):
        raise ValueError("训练快照 label map 指纹错误")
    if snapshot.get("preprocess_sha256") != json_hash(manifest["preprocess"]):
        raise ValueError("训练快照 preprocess 指纹错误")
    dataset = snapshot.get("dataset", {})
    hierarchical = dataset.get("hierarchical_input_ids")
    flat = dataset.get("flat_input_ids")
    target_labels = dataset.get("labels")
    if not isinstance(hierarchical, list) or not hierarchical or not (
        len(hierarchical) == len(flat) == len(target_labels)
    ):
        raise ValueError("训练数据快照长度错误")
    split = dataset.get("split", {})
    if set(split) != {"train", "validation", "test"}:
        raise ValueError("训练 split 字段错误")
    indices = split["train"] + split["validation"] + split["test"]
    if len(indices) != len(set(indices)) or sorted(indices) != list(range(len(hierarchical))):
        raise ValueError("训练 split 必须互斥且覆盖全部快照")
    hierarchy_tensor = torch.tensor(hierarchical, dtype=torch.long)
    flat_tensor = torch.tensor(flat, dtype=torch.long)
    if hierarchy_tensor.ndim != 3 or flat_tensor.ndim != 2:
        raise ValueError("训练输入维度错误")
    if hierarchy_tensor.numel() and (
        int(hierarchy_tensor.min()) < 0 or int(hierarchy_tensor.max()) >= config["vocab_size"]
    ):
        raise ValueError("训练层级 token 越界")
    reconstructed_flat = flatten_documents(hierarchy_tensor)
    if not torch.equal(reconstructed_flat, flat_tensor):
        raise ValueError("flat 训练快照不符合绑定的 flatten 规则")
    if any(not isinstance(label, int) or not 0 <= label < config["num_classes"] for label in target_labels):
        raise ValueError("训练 label 越界")
    expected_recipe = {
        "textcnn": {"optimizer": "Adam", "steps": 70, "lr": 0.025,
                    "clip_grad_norm": 2.0, "seed": SEED + 1,
                    "objective": "cross_entropy", "purpose": "controlled-overfit"},
        "han": {"optimizer": "Adam", "steps": 90, "lr": 0.025,
                "clip_grad_norm": 2.0, "seed": SEED + 2,
                "objective": "cross_entropy", "purpose": "controlled-overfit"},
    }[architecture]
    if snapshot.get("recipe") != expected_recipe:
        raise ValueError("训练 recipe 快照错误")


def package_classifier(model, architecture, vocab_spec, labels, subject,
                       artifact_id, version, training_snapshot):
    if architecture not in {"textcnn", "han"}:
        raise ValueError("架构不在白名单")
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    state_bytes = buffer.getvalue()
    manifest = {
        "schema": "document-classifier-v2", "artifact_id": artifact_id,
        "version": version, "subject": subject, "architecture": architecture,
        "config": copy.deepcopy(model.config), "vocab": copy.deepcopy(vocab_spec),
        "label_map": list(labels), "preprocess": expected_preprocess(architecture, model.config),
        "training_snapshot": copy.deepcopy(training_snapshot),
        "state_bytes_sha256": hashlib.sha256(state_bytes).hexdigest(),
        "state_tensor_sha256": classifier_tensor_state_hash(model.state_dict()),
    }
    validate_classifier_manifest(manifest)
    return {"manifest": manifest, "manifest_sha256": json_hash(manifest),
            "state_bytes": state_bytes}


vocab_spec = {"kind": "synthetic-id-list-v1", "tokens": CLASSIFIER_TOKENS, "pad_id": PAD}
label_names = ["技术", "财经", "体育"]
base_snapshot33 = {
    "dataset": {"name": "toy-document-classes-v1",
                "hierarchical_input_ids": documents.tolist(),
                "flat_input_ids": flat_documents.tolist(), "labels": document_labels.tolist(),
                "split": {"train": list(range(len(documents))), "validation": [], "test": []}},
    "vocab_sha256": json_hash(vocab_spec), "label_map_sha256": json_hash(label_names),
}

def training_snapshot_for(architecture, config):
    snapshot = copy.deepcopy(base_snapshot33)
    preprocess = expected_preprocess(architecture, config)
    snapshot["preprocess_sha256"] = json_hash(preprocess)
    snapshot["recipe"] = {
        "textcnn": {"optimizer": "Adam", "steps": 70, "lr": 0.025,
                    "clip_grad_norm": 2.0, "seed": SEED + 1,
                    "objective": "cross_entropy", "purpose": "controlled-overfit"},
        "han": {"optimizer": "Adam", "steps": 90, "lr": 0.025,
                "clip_grad_norm": 2.0, "seed": SEED + 2,
                "objective": "cross_entropy", "purpose": "controlled-overfit"},
    }[architecture]
    return snapshot


han_snapshot = training_snapshot_for("han", han.config)
textcnn_snapshot = training_snapshot_for("textcnn", textcnn.config)
han_package = package_classifier(
    han, "han", vocab_spec, label_names, "doc-team/demo",
    "han-demo", "1.0.0", han_snapshot,
)
textcnn_package = package_classifier(
    textcnn, "textcnn", vocab_spec, label_names, "doc-team/demo",
    "textcnn-demo", "1.0.0", textcnn_snapshot,
)
PUBLISHER_REGISTRY33 = MappingProxyType({
    ("han-demo", "1.0.0"): han_package["manifest_sha256"],
    ("textcnn-demo", "1.0.0"): textcnn_package["manifest_sha256"],
})


def trusted_classifier_load(package, expected_subject):
    if not isinstance(package, dict) or set(package) != {"manifest", "manifest_sha256", "state_bytes"}:
        raise ValueError("classifier package 字段错误")
    manifest = package["manifest"]
    if not isinstance(manifest, dict):
        raise ValueError("manifest 必须是字典")
    key = (manifest.get("artifact_id"), manifest.get("version"))
    expected_digest = PUBLISHER_REGISTRY33.get(key)
    if expected_digest is None:
        raise PermissionError("artifact id/version 未注册")
    computed_digest = json_hash(manifest)
    if package["manifest_sha256"] != computed_digest:
        raise ValueError("package 内 manifest hash 不一致")
    if computed_digest != expected_digest:
        raise PermissionError("package 内容不匹配发布者 registry")
    if manifest.get("subject") != expected_subject:
        raise PermissionError("业务主体不匹配")
    if hashlib.sha256(package["state_bytes"]).hexdigest() != manifest["state_bytes_sha256"]:
        raise ValueError("原始 state bytes 指纹不匹配")
    validate_classifier_manifest(manifest)
    state = torch.load(io.BytesIO(package["state_bytes"]), map_location="cpu", weights_only=True)
    if classifier_tensor_state_hash(state) != manifest["state_tensor_sha256"]:
        raise ValueError("tensor key/dtype/shape/bytes 指纹不匹配")
    constructors = {"textcnn": TextCNN, "han": HierarchicalAttentionNetwork}
    model = constructors[manifest["architecture"]](**manifest["config"])
    model.load_state_dict(state, strict=True)
    return model.eval(), manifest["label_map"]


restored_han, restored_labels = trusted_classifier_load(han_package, "doc-team/demo")
restored_textcnn, _ = trusted_classifier_load(textcnn_package, "doc-team/demo")
with torch.no_grad():
    assert torch.equal(han(documents, document_word_mask)["logits"],
                       restored_han(documents, document_word_mask)["logits"])
    assert torch.equal(textcnn(flat_documents, flat_mask)[0],
                       restored_textcnn(flat_documents, flat_mask)[0])
assert restored_labels == label_names

# 模型、完整 token 顺序、label map 即使同步重算内部 hash，也不能越过 registry。
forged_han = HierarchicalAttentionNetwork(**han.config)
with torch.no_grad():
    for parameter in forged_han.parameters():
        parameter.zero_()
fully_resigned = package_classifier(
    forged_han, "han", vocab_spec, label_names, "doc-team/demo",
    "han-demo", "1.0.0", han_snapshot,
)
resigned_vocab = copy.deepcopy(han_package)
resigned_vocab["manifest"]["vocab"]["tokens"][4:6] = list(reversed(
    resigned_vocab["manifest"]["vocab"]["tokens"][4:6]
))
resigned_vocab["manifest"]["training_snapshot"]["vocab_sha256"] = json_hash(
    resigned_vocab["manifest"]["vocab"]
)
resigned_vocab["manifest_sha256"] = json_hash(resigned_vocab["manifest"])
resigned_labels = copy.deepcopy(han_package)
resigned_labels["manifest"]["label_map"] = ["体育", "财经", "技术"]
resigned_labels["manifest"]["training_snapshot"]["label_map_sha256"] = json_hash(
    resigned_labels["manifest"]["label_map"]
)
resigned_labels["manifest_sha256"] = json_hash(resigned_labels["manifest"])
for candidate in (fully_resigned, resigned_vocab, resigned_labels):
    try:
        trusted_classifier_load(candidate, "doc-team/demo")
        raise AssertionError("整体重签模型/vocab/label 必须被 registry 拒绝")
    except PermissionError as exc:
        assert "registry" in str(exc)

bad_vocab = copy.deepcopy(vocab_spec)
bad_vocab["tokens"] = bad_vocab["tokens"][:-1]
for invalid_vocab, invalid_labels, phrase in [
    (bad_vocab, label_names, "vocab_size"),
    (vocab_spec, ["技术", "财经"], "label"),
    (vocab_spec, ["技术", "技术", "体育"], "label"),
]:
    try:
        package_classifier(
            han, "han", invalid_vocab, invalid_labels, "doc-team/demo",
            "invalid", "1.0.0", han_snapshot,
        )
        raise AssertionError("语义不一致制品必须在发布前拒绝")
    except ValueError as exc:
        assert phrase in str(exc), str(exc)

try:
    trusted_classifier_load(han_package, "other-team")
    raise AssertionError("跨主体加载不应通过")
except PermissionError:
    pass
print({"artifact_registry": dict(PUBLISHER_REGISTRY33),
       "resigned_model_vocab_label_rejected": True,
       "vocab_tokens": len(vocab_spec["tokens"]), "labels": label_names,
       "snapshot_documents": len(base_snapshot33["dataset"]["hierarchical_input_ids"])})

## 10. 常见失败模式与生产化清单

- TextCNN 对所有窗口直接 max，会把纯 padding 窗口的 bias 选中；必须先做窗口级 mask。
- HAN 将 padding 句送入普通 softmax 会得到均匀分布或 NaN；空行要返回全零权重，全空文档则拒绝。
- 反向 GRU 若先 `flip` 已 padding 序列却不同时处理长度，会让 padding 改变初始状态。
- 词表、分词、最大句数/词数、截断方向和标签顺序都属于模型输入合同，应和 checkpoint 原子发布。
- 长文档截断可能系统性漏掉结论段；上线前按长度分桶评估，并记录被截断比例。
- 监控预测分布、未知词率、长度分布、逐类延迟和漂移；回滚要同时回滚模型与预处理。
- 不把 attention 热力图当成合规解释；重要决策需独立解释与人工复核机制。

## 11. 原始论文与官方资料

- Kim, [Convolutional Neural Networks for Sentence Classification](https://arxiv.org/abs/1408.5882)：多窗口卷积与 max-over-time。
- Yang et al., [Hierarchical Attention Networks for Document Classification](https://aclanthology.org/N16-1174/)：词级/句级层次编码与注意力。
- Cho et al., [Learning Phrase Representations using RNN Encoder–Decoder](https://arxiv.org/abs/1406.1078)：GRU 门控思想。
- PyTorch 官方文档：[Conv1d](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)、[Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)。

本笔记复现的是结构与正确性合同，并未复现论文数据、词向量、调参预算或报告指标。